# Whisper baseline accuracy on a Colab GPU

This notebook is a thin wrapper around the same `whisper/scripts/run_baseline.py` used for local CPU runs - no script changes, just a GPU environment so larger checkpoints (e.g. `openai/whisper-large-v3`) finish in minutes instead of running for a very long time on a CPU-only laptop.

**Before running:**
- In Colab, go to `Runtime > Change runtime type` and select a GPU (T4 on the free tier).
- `ushahidi/dv-transcription-analysis` is a **private** repo, so cloning it here needs a GitHub Personal Access Token: create one at [github.com/settings/tokens](https://github.com/settings/tokens) (fine-grained, scoped to just this repo, with read-only "Contents" access is enough). The next cell prompts for it with `getpass` so it's typed once per session and never saved to disk or to this notebook file.

See `whisper/README.md` for the full pipeline explanation.

In [ ]:
from getpass import getpass

github_token = getpass("GitHub personal access token (read access to ushahidi/dv-transcription-analysis): ")
!git clone --branch main https://{github_token}@github.com/ushahidi/dv-transcription-analysis.git
%cd dv-transcription-analysis
del github_token  # don't keep it around in memory longer than needed

In [ ]:
# Colab ships a CUDA-enabled torch already - only the project's other deps are needed.
!pip install -q -r requirements.txt

## Pull the test set

This is a fresh clone, so `whisper/<language>/data/` (gitignored) doesn't exist yet - the test split needs to be downloaded again here, same as it was locally. `--max-samples` matches the local CPU run's default (20) so the WER is directly comparable to the existing `whisper-tiny` baseline results already committed under `whisper/<language>/results/`; raise it for a more robust GPU-scale eval.

In [ ]:
LANGUAGE = "kiswahili"  # or "english"
MODEL = "openai/whisper-large-v3"

!python whisper/scripts/prepare_dataset.py --language {LANGUAGE} --split test --max-samples 20

## Run the baseline

Reports WER + mean confidence and writes `whisper/<language>/results/baseline_<model>_test.{json,csv}`, exactly like the local CPU runs.

In [ ]:
!python whisper/scripts/run_baseline.py --language {LANGUAGE} --model {MODEL} --split test

## Download the results

Bring the JSON/CSV result files back to the local repo (e.g. to commit them alongside the existing `whisper-tiny` baseline results).

In [ ]:
from google.colab import files

safe_model = MODEL.split("/")[-1]
results_dir = f"whisper/{LANGUAGE}/results"
files.download(f"{results_dir}/baseline_{safe_model}_test.json")
files.download(f"{results_dir}/baseline_{safe_model}_test.csv")